In [ ]:
# Mod-12 Einzelmodul-Generator

# Mod-12 Einzelmodul-Generator

Erzeugt alle Zahlen ≤ n, deren Primfaktoren ausschließlich aus genau einer der mod-12 Klassen {1, 5, 7, 11} stammen.

**Funktionen:**
- Nutzt SageMath's optimierte Primzahl-Funktionen (`prime_range()`)
- Nutzt SageMath's optimierte Faktorisierung (`factor()`)
- Generiert CSV-Dateien sortiert nach Index und Wert

---

## 📋 Anleitung zur Ausführung

**Wichtig:** Dieses Notebook benötigt den **SageMath-Kernel**!

### Schritte:
1. **Stellen Sie sicher, dass der SageMath-Kernel aktiv ist** (oben rechts im Notebook)
2. **Führen Sie die Zellen der Reihe nach aus** (Shift+Enter für jede Zelle)
3. **Beginnen Sie mit Zelle 0** (Test-Ausgabe) - Sie sollten sofort Ausgabe sehen
4. **Führen Sie dann Zelle 2 aus** (SageMath-Imports)
5. **Führen Sie die restlichen Zellen nacheinander aus**

### Wenn keine Ausgabe erscheint:
- ✅ Prüfen Sie, ob der SageMath-Kernel aktiv ist (nicht Python!)
- ✅ Führen Sie die Zellen manuell aus (nicht "Run All")
- ✅ Prüfen Sie die Fehlerausgabe unter den Zellen
- ✅ Stellen Sie sicher, dass SageMath installiert ist: `sage --version`

### Ausgabe:
Das Programm erzeugt zwei CSV-Dateien:
- `mod12_single_module_by_index.csv` - sortiert nach Index-Muster
- `mod12_single_module_by_value.csv` - sortiert nach numerischem Wert

---

In [ ]:
# SageMath Imports - Nutze die schnellsten verfügbaren Bibliotheken
from sage.all import *
import csv
import sys

# SageMath's prime_range() ist deutlich schneller als manueller Sieb
# SageMath's Integer-Typ ist optimiert für große Zahlen
# factor() ist optimiert für große Zahlen
print("✓ SageMath erfolgreich geladen!")
print(f"SageMath Version: {version()}")
print(f"Python Version: {sys.version}")
sys.stdout.flush()  # Stelle sicher, dass die Ausgabe sofort angezeigt wird

## Konfiguration

In [ ]:
# Konfiguration
GROUP_ORDER = [1, 5, 7, 11]

# Beispiel: Obergrenze n
n = 1000
out_prefix = "mod12_single_module"

## Hilfsfunktionen

In [ ]:
def factorization_string(factor_dict):
    """
    Erzeugt eine schöne Faktorisierungsdarstellung wie 5^3*17*29^2.
    Nutzt SageMath's factor() Ergebnis.
    """
    parts = []
    # SageMath's factor() gibt ein Factorization-Objekt zurück
    # Wir konvertieren es zu sortierten Paaren (p, e)
    for p, e in sorted(factor_dict.items()):
        if e == 1:
            parts.append(str(p))
        else:
            parts.append(f"{p}^{e}")
    return "*".join(parts)

def factorization_string_from_sage_factor(fac):
    """
    Konvertiert SageMath's Factorization-Objekt zu String.
    """
    if isinstance(fac, Factorization):
        parts = []
        for p, e in sorted(fac):
            if e == 1:
                parts.append(str(p))
            else:
                parts.append(f"{p}^{e}")
        return "*".join(parts)
    else:
        # Fallback für dict
        return factorization_string(fac)

## Hauptfunktion: Generiere Zahlen für eine Restklasse

In [ ]:
def generate_group_numbers(n, primes, residue):
    """
    Generiere alle Zahlen <= n, deren Primfaktoren nur aus 'primes' stammen (eine Restklasse).
    Vermeidet Duplikate durch Erzwingen nicht-abnehmender Primzahl-Indizes in der Rekursion.
    
    Args:
        n: Obergrenze
        primes: Liste von Primzahlen (eine Restklasse mod 12)
        residue: Restklasse mod 12 (1, 5, 7 oder 11)
    
    Returns:
        Liste von Dictionaries mit Informationen über jede generierte Zahl
    """
    records = []
    
    # Precompute Indizes als 1-basierte Positionen innerhalb dieser Restklasse
    # primes[i] hat Index i+1
    def dfs(start_i, current_val, idx_list, fac_counter):
        # Speichere aktuelles Produkt wenn > 1 (1 ausschließen)
        if current_val > 1:
            # Nutze SageMath's factor() für saubere Faktorisierung
            # Aber wir haben bereits fac_counter, also nutzen wir das
            fac_str = factorization_string(fac_counter)
            
            records.append({
                "group_mod12": residue,
                "value": current_val,
                "indices": tuple(idx_list),  # wiederholte Indizes für Multiplizität
                "min_index": idx_list[0] if idx_list else None,
                "max_index": idx_list[-1] if idx_list else None,
                "len": len(idx_list),
                "factorization": fac_str,
            })
        
        # Versuche Multiplikation mit Primzahlen ab start_i (Wiederholungen erlauben)
        for i in range(start_i, len(primes)):
            p = primes[i]
            # Nutze SageMath Integer für präzise Arithmetik
            if current_val > n // p:
                break
            
            # Multipliziere mit p einmal
            idx_list.append(i + 1)  # 1-basierter Index in dieser Gruppe
            fac_counter[p] = fac_counter.get(p, 0) + 1
            dfs(i, current_val * p, idx_list, fac_counter)  # i (nicht i+1) => Wiederholungen erlauben
            # Backtrack
            fac_counter[p] -= 1
            if fac_counter[p] == 0:
                del fac_counter[p]
            idx_list.pop()
    
    dfs(0, 1, [], {})
    return records

## CSV-Schreibfunktion

In [ ]:
def write_csv(path, rows, fieldnames):
    """Schreibe CSV-Datei mit UTF-8 Kodierung."""
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        for r in rows:
            w.writerow(r)

## Hauptausführung

In [ ]:
# Prüfe Eingabe
if n < 1:
    raise ValueError("n muss >= 1 sein.")

print(f"Starte Berechnung für n = {n}...")
print(f"Nutze SageMath's optimierte Primzahl-Funktionen.\n")
sys.stdout.flush()

# Primzahlen bis n - Nutze SageMath's prime_range() (viel schneller!)
all_primes = prime_range(n + 1)
print(f"Gefunden: {len(all_primes)} Primzahlen <= {n}")
sys.stdout.flush()

# Gruppiere Primzahlen nach Restklasse mod 12 (nur 1, 5, 7, 11)
groups = {r: [] for r in GROUP_ORDER}
for p in all_primes:
    r = p % 12
    if r in groups:
        groups[r].append(p)

print("\nVerteilung nach Restklassen mod 12:")
for r in GROUP_ORDER:
    print(f"  Klasse {r}: {len(groups[r])} Primzahlen")
sys.stdout.flush()

In [ ]:
# Generiere Datensätze pro Gruppe
all_records = []
for r in GROUP_ORDER:
    ps = groups[r]
    if not ps:
        continue
    print(f"\nGeneriere Zahlen für Restklasse {r}...")
    sys.stdout.flush()
    records = generate_group_numbers(n, ps, r)
    all_records.extend(records)
    print(f"  {len(records)} Zahlen generiert")
    sys.stdout.flush()

print(f"\nGesamt: {len(all_records)} Zahlen <= {n} erzeugt")
sys.stdout.flush()

In [ ]:
# Füge zusätzliche Spalten für CSV-Lesbarkeit hinzu
for rec in all_records:
    # Indizes-Tupel -> String für CSV-Lesbarkeit
    rec["indices_str"] = " ".join(map(str, rec["indices"]))
    del rec["indices"]

# Ausgabe-Schema
fields = [
    "group_mod12",
    "value",
    "factorization",
    "indices_str",
    "len",
    "min_index",
    "max_index",
]

In [ ]:
# Sortierung 1: Nach Modul-Gruppenordnung, dann nach Index-Muster (lexikographisch), dann nach Wert
def parse_indices(s):
    """Parse Indizes-String zurück zu Tupel."""
    return tuple(int(x) for x in s.split()) if s else tuple()

group_rank = {r: i for i, r in enumerate(GROUP_ORDER)}

by_index = sorted(
    all_records,
    key=lambda rec: (
        group_rank.get(rec["group_mod12"], 999),
        parse_indices(rec["indices_str"]),
        rec["value"],
    )
)

# Sortierung 2: Nach numerischem Wert, dann Gruppe, dann Faktorisierung für Stabilität
by_value = sorted(
    all_records,
    key=lambda rec: (
        rec["value"],
        group_rank.get(rec["group_mod12"], 999),
        rec["factorization"],
    )
)

print("Sortierungen abgeschlossen.")
sys.stdout.flush()

In [ ]:
# Schreibe CSV-Dateien
out1 = f"{out_prefix}_by_index.csv"
out2 = f"{out_prefix}_by_value.csv"

write_csv(out1, by_index, fields)
write_csv(out2, by_value, fields)

print(f"\n✓ Fertig. {len(all_records)} Zahlen <= {n} erzeugt.")
print(f"CSV 1 (Index-Ordnung): {out1}")
print(f"CSV 2 (Wert-Ordnung):  {out2}")
sys.stdout.flush()

## Beispiel-Ausgabe

Zeige die ersten 20 Einträge (sortiert nach Wert):

In [ ]:
# Zeige Beispiel-Ausgabe - garantiert 20 Einträge
print("=" * 80)
print("Erste 20 Einträge (sortiert nach Wert):")
print("=" * 80)

# Versuche zuerst aus by_value Variable zu lesen (schneller)
if 'by_value' in globals() and len(by_value) > 0:
    num_to_show = min(20, len(by_value))
    print(f"\nZeige {num_to_show} von {len(by_value)} Einträgen:\n")
    for i, rec in enumerate(by_value[:20], 1):
        print(f"{i:3d}. Gruppe {rec['group_mod12']:2d} | Wert: {rec['value']:6d} | {rec['factorization']:30s} | Indizes: {rec['indices_str']}")
    sys.stdout.flush()
else:
    # Fallback: Versuche CSV zu lesen
    try:
        import pandas as pd
        df = pd.read_csv(out2)
        num_to_show = min(20, len(df))
        print(f"\nZeige {num_to_show} von {len(df)} Einträgen:\n")
        print(df.head(20).to_string(index=False))
        sys.stdout.flush()
    except (FileNotFoundError, ImportError, NameError):
        print("Warnung: Daten nicht verfügbar. Bitte führen Sie zuerst die vorherigen Zellen aus.")
        sys.stdout.flush()

print("\n" + "=" * 80)
sys.stdout.flush()